# Topic Modelling Tutorial

This tutorial does not cover all functions, but it's a starting point for further documentation.

Let's start by getting some sample data. We will use the Mallet sample_data folder.

In [1]:
# Python imports
import glob
from pathlib import Path
import pandas as pd

# Get a list of files in the data source directory
data_source_dir = "C:/mallet/mallet-2.0.8/sample-data/web/en"
data_files = glob.glob(data_source_dir + "/*.txt")
data_files = [Path(f).as_posix() for f in data_files]

# These files contain no metadata, so we will create a DataFrame with some extra information

In [ ]:
# Get some sample data into a dataframe

data_source_dir = "C:/mallet/mallet-2.0.8/sample-data/web/en"
data_files = glob.glob(data_source_dir + "/*.txt")
author_names = ["Abraham Alma", "Adele Aimee", "Adrian Albert", "Alfred Cooke",
                "Ailene Lott", "Allyson Alethea", "Alejando Albina", "Adelina Aimee",
                "Adele Alfredo", "Alex Aida", "Allen Liu", "Duncan Dubh"]
journals = [
    "PMLA", "NY Times", "Modern Philology", "Journal of the History of Ideas",
    "PMLA", "Journal of the History of Ideas", "Neuphilologische Mitteilungen", "Critical Inquiry",
    "PMLA", "NY Times", "Modern Philology", "Critical Inquiry"
]
volumes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
issues = [1, 1, 2, 1, 1, 2, 1, 1, 2, 2, 2, 2]
dates = [
    "2023-01-01", "2023-02-01", "2023-03-01", "2023-04-01",
    "2023-05-01", "2023-06-01", "2023-07-01", "2023-08-01",
    "2023-09-01", "2023-10-01", "2023-11-01", "2023-12-01"
]
pages = [
    "1-10", "11-20", "21-30", "31-40",
    "41-50", "51-60", "61-70", "71-80",
    "81-90", "91-100", "101-110", "111-120"
]
data = []
for i, file in enumerate(data_files):
    with open(file, "r", encoding="utf-8") as f:
        title = Path(file).stem.replace("_", " ").title()
        d = {}
        d["DOI"] = i
        d["title"] = title
        d["author"] = author_names[i]
        d["journal"] = journals[i]
        d["volume"] = volumes[i]
        d["issue"] = issues[i]
        d["date"] = dates[i]
        d["page"] = pages[i]
        d["text"] = f.read()
        d["doc_uri"] = file
        data.append(d)
df = pd.DataFrame(data)
df.head(5)

,DOI,title,author,journal,volume,issue,date,page,text,doc_uri
0,0,Elizabeth Needham,Abraham Alma,PMLA,1,1,2023-01-01,1-10,"Elizabeth Needham (died 3 May 1731), also know...",C:/mallet/mallet-2.0.8/sample-data/web/en\eliz...
1,1,Equipartition Theorem,Adele Aimee,NY Times,2,1,2023-02-01,11-20,The equipartition theorem is a formula from st...,C:/mallet/mallet-2.0.8/sample-data/web/en\equi...
2,2,Gunnhild,Adrian Albert,Modern Philology,3,2,2023-03-01,21-30,Gunnhild konungamóðir (mother of kings) or Gun...,C:/mallet/mallet-2.0.8/sample-data/web/en\gunn...
3,3,Hawes,Alfred Cooke,Journal of the History of Ideas,4,1,2023-04-01,31-40,Richard Hawes (1797–1877) was a United States ...,C:/mallet/mallet-2.0.8/sample-data/web/en\hawe...
4,4,Hill,Ailene Lott,PMLA,5,1,2023-05-01,41-50,Clem Hill (1877–1945) was an Australian cricke...,C:/mallet/mallet-2.0.8/sample-data/web/en\hill...


Now we want to get a dictionary with just our metadata.

In [3]:
# Get metadata dictionary
metadata_df = df[["DOI", "title", "author", "journal", "volume", "issue", "date", "page", "doc_uri"]]
metadata = metadata_df.to_dict(orient="records")
metadata[0]

{'DOI': 0,
 'title': 'Elizabeth Needham',
 'author': 'Abraham Alma',
 'journal': 'PMLA',
 'volume': 1,
 'issue': 1,
 'date': '2023-01-01',
 'page': '1-10',
 'doc_uri': 'C:/mallet/mallet-2.0.8/sample-data/web/en\\elizabeth_needham.txt'}

Similarly, we will get our training data (texts) from the text column in the dataframe.

In [4]:
training_data = df.text.tolist()
training_data[0]

"Elizabeth Needham (died 3 May 1731), also known as Mother Needham, was an English procuress and brothel-keeper of 18th-century London, who has been identified as the bawd greeting Moll Hackabout in the first plate of William Hogarth's series of satirical etchings, A Harlot's Progress. Although Needham was notorious in London at the time, little is recorded of her life, and no genuine portraits of her survive. Her house was the most exclusive in London and her customers came from the highest strata of fashionable society, but she eventually crossed the moral reformers of the day and died as a result of the severe treatment she received after being sentenced to stand in the pillory.\n"

We're now ready to begin the topic modelling process.

We start by importing the Lexos Mallet module. This module is a wrapper around the Mallet topic modelling library, which is written in Java. The Lexos Mallet module provides a Python interface to the Mallet library, allowing us to use it within our Python code.

In [5]:
# Import the Mallet class from lexos.topic_modeling.mallet
from lexos.topic_modeling.mallet import Mallet

Next we instantiate a `Mallet` object and import our data. The `Mallet` class makes a copy of your data in a single `data.txt` file. This is the source of "ground truth" for your data. When you designate a path for the `data.txt` file, a parent directory is created if it does not already exist. This is the "model directory". When you import the data, a `formatted_training_data` file is created inside the model directory. This is a version of `data.txt` that has been formatted for use by the Mallet Java package.

In [6]:
# Designate a path to the training data. This is where the data will be imported into Mallet.
path_to_training_data = "C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/data.txt"

# Instantiate the Mallet class and import the data. Wherever you import it is the model directory.
mallet = Mallet()

# Import the data into Mallet
mallet.import_data(training_data, path_to_training_data)

Now we can train the model. We can set any number of parameters, but the most important ones are `num_topics`, the path to our formatted training data file, and the paths to save the output. If you give filenames, these will be relative to the model directory; otherwise, you can use absolute paths if you wish to save output in a different location. By default, the Mallet progress will be printed to the console. If you do not want to see that, set `verbose=False`.

In [ ]:
# Train the model
mallet.train(
    num_topics=10,
    path_to_formatted_training_data="formatted_training_data.mallet",
    path_to_model="model_10",
    path_to_state="model_10_state",
    path_to_topic_keys="keys.txt",
    path_to_topic_distributions="distributions.txt",
    path_to_term_weights="term_weights.txt",
    verbose=False
)


Output()

You can access lots of information about your model with `mallet.metadata`.

In [8]:
mallet.metadata

{'model_directory': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment',
 'path_to_training_data': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/data.txt',
 'path_to_formatted_training_data': 'formatted_training_data.mallet',
 'num_docs': 12,
 'mean_num_tokens': 205.66666666666666,
 'vocab_size': 1247,
 'num_topics': 10,
 'optimize_interval': 10,
 'path_to_model': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/model_10',
 'path_to_state': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/model_10_state',
 'path_to_topic_keys': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/keys.txt',
 'path_to_topic_distributions': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/distributions.txt',
 'path_to_term_weights': 'C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/term_weights.txt',
 'path_to_diagnostics': None,
 'training_command': 'C:/ma

### Generating a Dfr-Browser



In [11]:
# Import the Lexos DfrBrowser class
from lexos.topic_modeling.dfr_browser import DfrBrowser

In [12]:
# Designate a path for the browser directory. This is where the browser will be created.
path_to_browser_dir = f"{mallet.metadata['model_directory']}/dfr_browser"

# Designate a path to the state file. Since we have a Mallet instance in memory, we'll use its state file.
path_to_state_file = mallet.metadata["path_to_state"]

# We'll also get the number of topics from the Mallet instance

# Instantiate the DfrBrowser class
browser = DfrBrowser(
    path_to_browser_dir = path_to_browser_dir,
    metadata = metadata, # We created this dict above
    num_topics = mallet.metadata["num_topics"],
    path_to_state_file = path_to_state_file
)


We now need to build the browser. The output tells us what extra files were saved in the browser directory.

In [13]:
browser.build()

beta value, not saved in a file: 0.01
Wrote topic-words information to C:\Users\scott\Documents\uv_lexos\src\lexos\topic_modeling\experiment\dfr_browser\data/tw.json
Wrote sparse doc-topics to C:\Users\scott\Documents\uv_lexos\src\lexos\topic_modeling\experiment\dfr_browser\data/dt.json.zip
Created stub file in C:/Users/scott/Documents/uv_lexos/src/lexos/topic_modeling/experiment/dfr_browser/data/info.json


If building was successful, we can now open the browser. We can do this directly by calling `browser.serve()`, which will start a localhost on port 8888. If you are using a Jupyter notebook, or the port is occupied, you can change it with something like `browser.serve(port=5000)`. Once the server starts, a browser will open with the DfrBrowser running in it. To stop the server, type Control+C, or, if you served the browser from a Jupyter notebook, use the notebook's "Interrupt" button.

Alternatively, you can skip `browser.serve()` by opening a terminal, `cd`ing to the browser directory and starting a localhost with something like `python -m http.server`. Finally, if you are running a remote server, you can simply copy the `dfr_browser` directory to the appropriate location on the server.

In [ ]:
browser.serve()